> # CHIBI FROM CONCEPT — da concept art ao chibi, sem passar pelo Git
>
> **Fluxo completo, uma personagem por execucao:**
>
> ```
> upload da concept art
>     -> FLOW 01 (deterministico, local)  -> source/ + reference/
>     -> FLUX.2 klein 4B                  -> chibi
>     -> 2 ZIPs
> ```
>
> **Por que existe:** o `flux2_klein_4b_eval.ipynb` exige que a personagem
> ja esteja versionada em `characters/<id>/reference/`. Isso obriga a subir
> arte pesada para o Git (hoje via LFS, que este clone nem resolve: os PNGs
> da waifu_002/003 sao ponteiros de 132 bytes). Aqui a arte entra por
> **upload** e nada precisa ser commitado.
>
> **O que este notebook NAO faz:**
>
> - nao substitui o `flux2_klein_4b_eval.ipynb`, que continua sendo o
>   baseline historico das Runs 001/002/003 da waifu_001 — **intocado**;
> - nao aprova nada. O resultado sai marcado `EXPERIMENTAL` e a avaliacao
>   artistica continua sendo humana;
> - nao inventa design: o FLOW 01 e deterministico e os recortes do
>   identity kit sao heuristicos, sempre `[HUMAN REVIEW REQUIRED]`.
>
> **Saida — dois ZIPs, como pedido:**
>
> | ZIP | conteudo | para que serve |
> |---|---|---|
> | `<id>_character_kit.zip` | `source/` + `reference/` + `character.yaml` | e o que iria para o Git; reutilizavel nos outros notebooks |
> | `<id>_chibi_result.zip` | chibi + `recipe.json` + workflow resolvido + hashes + logs | o resultado desta execucao |


---

In [ ]:
#@title 0. Painel { display-mode: "form" }
#@markdown Identificador da personagem. Vira o nome da pasta e aparece nos
#@markdown recipes — use algo estavel (`waifu_004`, `elara`...).
CHARACTER_ID = "waifu_004"  #@param {type:"string"}
DISPLAY_NAME = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### PROMPT
#@markdown - `generico`: reutilizavel para qualquer personagem. A identidade
#@markdown   vem das imagens de referencia, nunca do texto. **Recomendado.**
#@markdown - `auto_from_image`: um tagger le a arte e descreve a personagem.
#@markdown   Sai do prompt generico — o resultado deixa de ser comparavel
#@markdown   entre personagens. Requer revisao humana antes de usar.
#@markdown - `manual`: voce escreve.
PROMPT_MODE = "generico"  #@param ["generico", "auto_from_image", "manual"]
PROMPT_MANUAL = ""  #@param {type:"string"}
#@markdown Confianca minima do tagger (so no modo auto).
TAGGER_THRESHOLD = 0.35  #@param {type:"slider", min:0.1, max:0.9, step:0.05}

#@markdown ---
#@markdown ### GERACAO
SEED = 42  #@param {type:"integer"}
WORKFLOW_VERSION = "v2"  #@param ["v1", "v2"]
#@markdown v2 = multi-referencia (full_body + face + outfit), a configuracao
#@markdown da Run 003. v1 usa so a full_body.
USAR_MULTI_REFERENCIA = True  #@param {type:"boolean"}

import pathlib, datetime

REPO_BRANCH = "arena/01a07ece-chibicreate"
REPO = pathlib.Path("/content/ChibiCreate")
UPLOAD_DIR = pathlib.Path("/content/concept_upload")
STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

assert CHARACTER_ID.strip(), "CHARACTER_ID nao pode ser vazio"
assert "/" not in CHARACTER_ID, "CHARACTER_ID e nome de pasta, sem barras"
if PROMPT_MODE == "manual":
    assert PROMPT_MANUAL.strip(), (
        "PROMPT_MODE='manual' exige PROMPT_MANUAL preenchido.")

# Prompt-base GENERICO: descreve a TRANSFORMACAO, nunca a personagem.
# Proibido citar cor de cabelo/olhos, chifres, roupa, capa ou acessorio —
# isso viria do texto em vez da arte e quebraria a reutilizacao.
PROMPT_GENERICO = (
    "Transform this character into a clean stylized chibi full-body "
    "character, preserving the same identity, the same outfit and the same "
    "accessories as shown in the reference images. Simple cel shading, "
    "clean lineart, plain background.")

print("personagem :", CHARACTER_ID)
print("prompt     :", PROMPT_MODE)
print("seed       :", SEED, "| workflow", WORKFLOW_VERSION)
if PROMPT_MODE == "auto_from_image":
    print()
    print("[ATENCAO] prompt automatico descreve ESTA personagem: o texto")
    print("  deixa de ser reutilizavel e o resultado nao e comparavel com")
    print("  execucoes de prompt generico. Sera registrado no recipe como")
    print("  character_specific_prompt=true.")


---
## 1. Repositorio e ambiente

In [ ]:
#@title 1. Repositorio { display-mode: "form" }
import subprocess, sys

if REPO.exists():
    subprocess.run(["git","-C",str(REPO),"fetch","-q","origin",REPO_BRANCH], check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","-q","-B",REPO_BRANCH,
                    "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git","clone","-q","--branch",REPO_BRANCH,
                    "https://github.com/BloomRX/ChibiCreate.git", str(REPO)],
                   check=True)

SCRIPTS = REPO / "scripts"
if not (SCRIPTS / "chibi" / "flow01.py").exists():
    raise SystemExit(
        f"BLOCKED — {SCRIPTS}/chibi nao existe. A branch '{REPO_BRANCH}' foi "
        "baixada? A main do repositorio so tem o README.")
sys.path.insert(0, str(SCRIPTS))

print("commit:", subprocess.run(["git","-C",str(REPO),"rev-parse","HEAD"],
      capture_output=True, text=True).stdout.strip())
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import sys, json

WEIGHTS_GB = 7.75      # flux-2-klein-4b.safetensors
MIN_VRAM_GB = 13.0     # model card oficial: '~13GB VRAM'

try:
    import torch
except ImportError:
    torch = None

print('=' * 62)
print('AMBIENTE COLAB — detectado, nao presumido')
print('=' * 62)
print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__ if torch else 'ausente')

if torch is None or not torch.cuda.is_available():
    print('CUDA   : INDISPONIVEL')
    print('COLAB_GPU_INSUFFICIENT — nenhuma GPU CUDA.')
    print('Runtime -> Alterar tipo de ambiente de execucao -> GPU')
    raise SystemExit('FASE 3B permanece BLOCKED')

props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / 1024 ** 3
free_gb = torch.cuda.mem_get_info()[0] / 1024 ** 3

GPU_INFO = {
    'name': props.name,
    'vram_total_gb': round(total_gb, 2),
    'vram_free_gb': round(free_gb, 2),
    'cuda': torch.version.cuda,
    'capability': '{}.{}'.format(props.major, props.minor),
    'torch': torch.__version__,
    'python': sys.version.split()[0],
    'bf16_supported': props.major >= 8,
}
print('GPU    :', GPU_INFO['name'])
print('VRAM   : {:.2f} GB total / {:.2f} GB livre'.format(total_gb, free_gb))
print('CUDA   :', GPU_INFO['cuda'], '| capability', GPU_INFO['capability'])
print('Necessario : ~{:.0f} GB'.format(MIN_VRAM_GB))
print()

if not GPU_INFO['bf16_supported']:
    print('AVISO: sem bf16 nativo (capability < 8.0, ex. T4).')
    print('O ComfyUI cai para fp16. Pode funcionar; registre o resultado.')
    print()

if total_gb < MIN_VRAM_GB:
    print('=' * 62)
    print('COLAB_GPU_INSUFFICIENT')
    print('=' * 62)
    print('{} tem {:.1f} GB; o FLUX.2 klein 4B precisa de ~{:.0f} GB.'
          .format(GPU_INFO['name'], total_gb, MIN_VRAM_GB))
    print('PARE. Nao usar CPU, nao trocar de modelo, nao quantizar por conta.')
    raise SystemExit('COLAB_GPU_INSUFFICIENT')

print('GPU ADEQUADA — pode prosseguir.')
json.dump(GPU_INFO, open('/content/gpu_info.json', 'w'), indent=2)


---
## 2. Upload da concept art

Envie **uma** imagem de corpo inteiro da personagem. Ela e a unica entrada
humana deste notebook: todo o resto e derivado dela.

Recomendado: fundo simples ou transparente, personagem inteira visivel. O
FLOW 01 isola o sujeito e normaliza para 1024x1024, mas nao inventa o que
estiver cortado.

In [ ]:
#@title 2. Upload da concept art { display-mode: "form" }
import shutil
from PIL import Image

try:
    from google.colab import files
    subidos = files.upload()
    for nome, dados in subidos.items():
        (UPLOAD_DIR / nome).write_bytes(dados)
        print("recebido:", nome, len(dados), "bytes")
except Exception as e:
    print("[fora do Colab] copie a arte manualmente para", UPLOAD_DIR, "|", e)

_imgs = [p for p in sorted(UPLOAD_DIR.iterdir())
         if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")]
if not _imgs:
    raise SystemExit(f"BLOCKED — nenhuma imagem em {UPLOAD_DIR}.")
if len(_imgs) > 1:
    print()
    print("[aviso] mais de uma imagem enviada. O FLOW 01 escolhe a de maior")
    print("        area como principal; as outras entram como fontes extras.")

CONCEPT = max(_imgs, key=lambda p: Image.open(p).size[0] * Image.open(p).size[1])
_im = Image.open(CONCEPT)
print()
print("arte principal:", CONCEPT.name, _im.size, _im.mode)
display(_im.copy().resize((min(384, _im.width),
                           int(_im.height * min(384, _im.width) / _im.width))))


---
## 3. FLOW 01 — gerar o que estaria no Git

Deterministico: sem IA, sem aleatoriedade, mesma entrada = mesma saida. Cria
`source/` e `reference/` (`full_body`, `face`, `hair`, `outfit`, `palette`,
`sheet`) — exatamente a estrutura que os outros notebooks esperam.

In [ ]:
#@title 3. FLOW 01 { display-mode: "form" }
#@markdown Refazer do zero se a personagem ja existir nesta sessao.
FORCAR = False  #@param {type:"boolean"}
import subprocess, shutil, pathlib

CHAR_DIR = REPO / "characters" / CHARACTER_ID
if CHAR_DIR.exists() and FORCAR:
    shutil.rmtree(CHAR_DIR)

if not CHAR_DIR.exists():
    cmd = [sys.executable, "-m", "scripts.chibi.cli", "character", "new",
           CHARACTER_ID]
    if DISPLAY_NAME.strip():
        cmd += ["--name", DISPLAY_NAME.strip()]
    r = subprocess.run(cmd, cwd=str(REPO), capture_output=True, text=True)
    print(r.stdout[-600:] or r.stderr[-600:])

# A arte enviada vira a source da personagem — e o unico ponto onde um
# arquivo humano entra no fluxo.
SOURCE_DIR = CHAR_DIR / "source"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
DEST = SOURCE_DIR / f"{CHARACTER_ID}{CONCEPT.suffix.lower()}"
shutil.copy2(CONCEPT, DEST)
print("source:", DEST.relative_to(REPO))

_flow = subprocess.run(
    [sys.executable, "-m", "scripts.chibi.cli", "flow01", CHARACTER_ID]
    + (["--force"] if FORCAR else []),
    cwd=str(REPO), capture_output=True, text=True)
print(_flow.stdout[-2500:])
if _flow.returncode != 0:
    print(_flow.stderr[-1500:])
    raise SystemExit("BLOCKED — FLOW 01 falhou (saida acima).")

REF_DIR = CHAR_DIR / "reference"
_esperados = ["full_body.png", "face.png", "hair.png", "outfit.png",
              "palette.json", "sheet.png"]
_faltando = [n for n in _esperados if not (REF_DIR / n).exists()]
if _faltando:
    raise SystemExit(f"BLOCKED — FLOW 01 nao gerou: {_faltando}")
print("reference/ OK:", ", ".join(_esperados))


In [ ]:
#@title 3b. Conferir o identity kit { display-mode: "form" }
#@markdown Os recortes sao HEURISTICOS (fracoes da caixa do sujeito). Se
#@markdown estiverem errados, ajuste `reference_regions` no `character.yaml`
#@markdown e rode a celula 3 de novo com FORCAR marcado.
import matplotlib.pyplot as plt
from PIL import Image

_nomes = ["full_body.png", "face.png", "hair.png", "outfit.png"]
fig, ax = plt.subplots(1, len(_nomes), figsize=(4.6 * len(_nomes), 5))
for a, n in zip(ax, _nomes):
    a.imshow(Image.open(REF_DIR / n))
    a.set_title(n, fontsize=11)
    a.axis("off")
fig.tight_layout()
plt.show()

print("[HUMAN REVIEW REQUIRED] os recortes pegaram as regioes certas?")
print("  face = rosto | hair = cabelo/cabeca | outfit = corpo e roupa")
print("  Recorte errado aqui degrada a referencia que o FLUX vai consumir.")


---
## 4. Prompt

In [ ]:
#@title 4. Definir o prompt { display-mode: "form" }
import json, sys
sys.path.insert(0, str(SCRIPTS))
from chibi.model_registry import termos_especificos_no_prompt

TAGGER_INFO = None

if PROMPT_MODE == "generico":
    PROMPT = PROMPT_GENERICO
    PROMPT_SOURCE = "preset:chibi_transform_generic_v1"
    CHARACTER_SPECIFIC = False

elif PROMPT_MODE == "manual":
    PROMPT = PROMPT_MANUAL.strip()
    PROMPT_SOURCE = "manual"
    CHARACTER_SPECIFIC = bool(termos_especificos_no_prompt(PROMPT))

else:  # auto_from_image
    # WD14 SwinV2 v3 (SmilingWolf), Apache-2.0, ONNX em CPU. E o tagger
    # consagrado para arte anime; roda sem GPU e sem chave de API.
    # Ele NAO "entende" a personagem: devolve tags Danbooru por confianca.
    !pip install -q onnxruntime huggingface_hub pandas
    import numpy as np, pandas as pd, onnxruntime
    from huggingface_hub import hf_hub_download
    from PIL import Image

    TAGGER_REPO = "SmilingWolf/wd-swinv2-tagger-v3"
    _onnx = hf_hub_download(TAGGER_REPO, "model.onnx")
    _csv = hf_hub_download(TAGGER_REPO, "selected_tags.csv")
    _tags = pd.read_csv(_csv)

    _sess = onnxruntime.InferenceSession(_onnx,
                                         providers=["CPUExecutionProvider"])
    _, _h, _w, _ = _sess.get_inputs()[0].shape
    _img = Image.open(REF_DIR / "full_body.png").convert("RGBA")
    _canvas = Image.new("RGBA", _img.size, (255, 255, 255, 255))
    _canvas.alpha_composite(_img)
    _img = _canvas.convert("RGB")
    _lado = max(_img.size)
    _quad = Image.new("RGB", (_lado, _lado), (255, 255, 255))
    _quad.paste(_img, ((_lado - _img.width) // 2, (_lado - _img.height) // 2))
    _arr = np.asarray(_quad.resize((_w, _h), Image.BICUBIC),
                      dtype=np.float32)[:, :, ::-1]   # o modelo espera BGR
    _probs = _sess.run(None, {_sess.get_inputs()[0].name: _arr[None]})[0][0]

    _geral = _tags.index[_tags["category"] == 0]
    _achadas = [(_tags["name"][i], float(_probs[i])) for i in _geral
                if _probs[i] >= TAGGER_THRESHOLD]
    _achadas.sort(key=lambda x: -x[1])
    _desc = ", ".join(t.replace("_", " ") for t, _ in _achadas[:25])

    PROMPT = (f"Transform this character into a clean stylized chibi "
              f"full-body character, preserving the same identity. "
              f"The character has: {_desc}. Simple cel shading, clean "
              f"lineart, plain background.")
    PROMPT_SOURCE = f"auto:{TAGGER_REPO}@threshold={TAGGER_THRESHOLD}"
    CHARACTER_SPECIFIC = True
    TAGGER_INFO = {"model": TAGGER_REPO, "license": "Apache-2.0",
                   "threshold": TAGGER_THRESHOLD,
                   "tags": [{"tag": t, "confidence": round(c, 4)}
                            for t, c in _achadas[:25]]}
    print("tags detectadas (confianca):")
    for t, c in _achadas[:25]:
        print(f"  {c:.3f}  {t}")
    print()
    print("[HUMAN REVIEW REQUIRED] o tagger erra. Confira se a descricao")
    print("  bate com a arte; tag errada vira design errado no chibi.")

print()
print("=" * 68)
print("PROMPT:", PROMPT)
print("origem:", PROMPT_SOURCE, "| especifico da personagem:",
      CHARACTER_SPECIFIC)
print("=" * 68)

_termos = termos_especificos_no_prompt(PROMPT)
if _termos and not CHARACTER_SPECIFIC:
    raise SystemExit(
        f"BLOCKED — o prompt marcado como generico contem termos de "
        f"personagem: {_termos}. A identidade deve vir das imagens.")
if _termos:
    print()
    print("termos especificos no prompt:", _termos)


---
## 5. Modelo e ComfyUI

In [ ]:
from huggingface_hub import hf_hub_download
import pathlib, shutil

M = pathlib.Path('/content/ComfyUI/models')
REPO = 'Comfy-Org/vae-text-encorder-for-flux-klein-4b'
REV  = '5f526678002e43af5551dadb73ce2e8c91b43afe'

DOWNLOADS = [
    ('split_files/diffusion_models/flux-2-klein-4b.safetensors',
     M / 'diffusion_models',
     'ec3d4e733a771f61c052fb4856c48b336c55eaf2c65487c2a1faeb9bbda7a343'),
    ('split_files/text_encoders/qwen_3_4b.safetensors',
     M / 'text_encoders',
     '6c671498573ac2f7a5501502ccce8d2b08ea6ca2f661c458e708f36b36edfc5a'),
    # Alternativa fp4 para GPU apertada — troque a linha acima por esta:
    # ('split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors',
    #  M / 'text_encoders',
    #  '3eab03a77adb0ee5304a4e677d5c10ac22f9049c1d7c894adca4f8bb39206ca8'),
    ('split_files/vae/flux2-vae.safetensors',
     M / 'vae',
     '868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3'),
]

MODEL_RECORD = []
for remote, dest, expected in DOWNLOADS:
    dest.mkdir(parents=True, exist_ok=True)
    fname = remote.split('/')[-1]
    target = dest / fname
    if target.exists():
        print('ja existe:', fname)
    else:
        print('baixando :', fname)
        got = hf_hub_download(repo_id=REPO, revision=REV, filename=remote)
        shutil.copy(got, target)
    MODEL_RECORD.append({'file': fname, 'repo': REPO, 'revision': REV,
                         'license': 'Apache-2.0',
                         'size_bytes': target.stat().st_size,
                         'sha256_expected': expected})
!df -h /content | tail -1


In [ ]:
import hashlib, pathlib, json

def sha256_of(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

ok = True
for rec in MODEL_RECORD:
    hit = list(pathlib.Path('/content/ComfyUI/models').rglob(rec['file']))[0]
    actual = sha256_of(hit)
    rec['sha256_actual'] = actual
    rec['sha256_match'] = actual == rec['sha256_expected']
    ok &= rec['sha256_match']
    print(('OK   ' if rec['sha256_match'] else 'FALHA'), rec['file'])

json.dump(MODEL_RECORD, open('/content/model_record.json', 'w'), indent=2)
if not ok:
    raise SystemExit('SHA256 divergente — PARE e reporte.')
print('Pesos conferem com models.lock.yaml.')


In [ ]:
import subprocess, time, urllib.request, json

LOG = open('/content/comfyui.log', 'w')
proc = subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1',
                         '--port', '8188'],
                        cwd='/content/ComfyUI', stdout=LOG,
                        stderr=subprocess.STDOUT)

print('subindo ComfyUI...')
for i in range(120):
    time.sleep(5)
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats',
                                    timeout=5) as r:
            stats = json.load(r)
        print('no ar apos ~{}s'.format((i + 1) * 5))
        break
    except Exception:
        if proc.poll() is not None:
            print(open('/content/comfyui.log').read()[-3000:])
            raise SystemExit('ComfyUI morreu ao iniciar')
else:
    print(open('/content/comfyui.log').read()[-3000:])
    raise SystemExit('ComfyUI nao respondeu em 10 min')

print(json.dumps(stats.get('system', {}), indent=2))


In [ ]:
import os
os.environ['CHIBI_COMFY_URL'] = 'http://127.0.0.1:8188'
%cd /content/ChibiCreate

!python -m scripts.chibi.cli comfy status --env colab_flux2
print('=' * 62)
!python -m scripts.chibi.cli comfy preflight --env colab_flux2
print('=' * 62)
!python -m scripts.chibi.cli comfy validate --env colab_flux2 --workflow experimental/flux2_klein_edit


---
## 6. Gerar o chibi

Mesma CLI do notebook de baseline, mesmo motor. A diferenca e a origem da
personagem: aqui ela veio da concept art enviada, nao do Git.

In [ ]:
#@title 6. Executar { display-mode: "form" }
import subprocess, json, pathlib

_refs = []
if USAR_MULTI_REFERENCIA:
    # Configuracao da Run 003: full_body (principal) + face + outfit.
    for _r in ("reference/face.png", "reference/outfit.png"):
        _refs += ["--ref", _r]

CMD = [sys.executable, "-m", "scripts.chibi.cli", "experiment", "model-eval",
       "--model", "flux2-klein", "--character", CHARACTER_ID,
       "--seed", str(SEED), "--prompt", PROMPT,
       "--workflow-version", WORKFLOW_VERSION] + _refs

print("$", " ".join(CMD[2:]))
print()
_run = subprocess.run(CMD, cwd=str(REPO), capture_output=True, text=True)
print(_run.stdout[-3000:])
if _run.returncode != 0:
    print(_run.stderr[-2000:])
    raise SystemExit("BLOCKED — a geracao falhou (saida acima).")

EVAL_ROOT = REPO / "experiments" / "model_eval" / "flux2_klein_4b"
RUNS = sorted(p for p in EVAL_ROOT.glob("run_*") if p.is_dir())
RUN_DIR = RUNS[-1]
print("run:", RUN_DIR.relative_to(REPO))


In [ ]:
#@title 6b. Resultado { display-mode: "form" }
import matplotlib.pyplot as plt
from PIL import Image

RECIPE = json.loads((RUN_DIR / "recipe.json").read_text())
OUT = RUN_DIR / (RECIPE.get("output_filename") or "output.png")
if not OUT.exists():
    _cands = sorted(RUN_DIR.glob("*.png"))
    _cands = [p for p in _cands if p.name != "input.png"]
    if not _cands:
        raise SystemExit(f"BLOCKED — nenhuma imagem de saida em {RUN_DIR}")
    OUT = _cands[-1]

fig, ax = plt.subplots(1, 2, figsize=(11, 5.6))
for a, p, t in ((ax[0], REF_DIR / "full_body.png", "CONCEPT (normalizada)"),
                (ax[1], OUT, "CHIBI (FLUX.2 klein)")):
    a.imshow(Image.open(p)); a.set_title(t); a.axis("off")
fig.tight_layout(); plt.show()

print("tempo:", RECIPE.get("execution_time"), "| seed:", RECIPE.get("seed"))
print("output sha256:", RECIPE.get("output_sha256"))
print()
print("[HUMAN REVIEW REQUIRED] EXPERIMENTAL — nao e Chibi Master.")
print("  A avaliacao artistica e humana: nenhuma metrica aqui diz se ficou bom.")


---
## 7. Os dois ZIPs

In [ ]:
#@title 7. Empacotar { display-mode: "form" }
import zipfile, hashlib, json, shutil, pathlib

def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

# ---- ZIP 1: character kit (o que iria para o Git) --------------------
KIT = pathlib.Path(f"/content/{CHARACTER_ID}_character_kit.zip")
_hashes_kit = {}
with zipfile.ZipFile(KIT, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(CHAR_DIR.rglob("*")):
        if not p.is_file() or ".ipynb_checkpoints" in p.parts:
            continue
        rel = p.relative_to(CHAR_DIR.parent)
        z.write(p, rel)
        _hashes_kit[str(rel)] = _sha(p)
    z.writestr(f"{CHARACTER_ID}/hashes.json",
               json.dumps(_hashes_kit, indent=2, ensure_ascii=False))
    z.writestr(f"{CHARACTER_ID}/README.md", f"""# {CHARACTER_ID} — character kit

Gerado por `notebooks/chibi_from_concept.ipynb` em {STAMP}.

`source/` e a arte enviada; `reference/` foi derivada dela pelo FLOW 01,
que e deterministico (sem IA, sem aleatoriedade).

Para reusar nos outros notebooks, descompacte em `characters/`.

Os recortes do identity kit sao heuristicos: **[HUMAN REVIEW REQUIRED]**.
""")

# ---- ZIP 2: resultado da execucao ------------------------------------
RES = pathlib.Path(f"/content/{CHARACTER_ID}_chibi_result.zip")
_extra = {
    "character_id": CHARACTER_ID,
    "created": STAMP,
    "prompt": PROMPT,
    "prompt_mode": PROMPT_MODE,
    "prompt_source": PROMPT_SOURCE,
    "character_specific_prompt": CHARACTER_SPECIFIC,
    "tagger": TAGGER_INFO,
    "concept_upload": {"filename": CONCEPT.name, "sha256": _sha(CONCEPT)},
    "flow01": "deterministico, local, sem IA generativa",
    "multi_reference": USAR_MULTI_REFERENCIA,
    "workflow_version": WORKFLOW_VERSION,
    "approval_status": "experimental",
    "note": ("Origem da personagem: concept art enviada no notebook, nao o "
             "Git. Nao substitui o baseline flux2_klein_4b_eval.ipynb."),
}
with zipfile.ZipFile(RES, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUN_DIR.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(RUN_DIR))
    z.write(REF_DIR / "full_body.png", "concept_normalized.png")
    z.write(CONCEPT, f"concept_original{CONCEPT.suffix.lower()}")
    z.writestr("notebook_context.json",
               json.dumps(_extra, indent=2, ensure_ascii=False))

for _z in (KIT, RES):
    print(f"{_z}  ({_z.stat().st_size/1e6:.1f} MB)")

try:
    from google.colab import files as _f
    _f.download(str(KIT)); _f.download(str(RES))
except Exception as e:
    print("[fora do Colab] baixe manualmente |", e)
